# 🔥 Predictive Workforce Wellbeing & Burnout Risk Analytics
### End-to-End ML Pipeline — Google Colab Edition

| Detail | Info |
|---|---|
| **Project** | Workforce Burnout Risk Analytics Platform |
| **GitHub** | [ShivanshuRai07/Burnout](https://github.com/ShivanshuRai07/Burnout) |
| **Models** | HistGradientBoostingRegressor + CalibratedClassifierCV |
| **Dataset Size** | ~178,750 synthetic employee records |
| **Runtime** | CPU or GPU (T4 recommended) |

---

## 📋 Notebook Sections
1. 📦 Install & Import Libraries  
2. 🗃️ Generate Synthetic Dataset  
3. 🔧 Feature Engineering  
4. 📊 Exploratory Data Analysis (EDA)  
5. ✂️ Preprocessing & Train/Test Split  
6. 🎯 Task A — Burnout Score Regression  
7. 🚨 Task B — Elevated Risk Classification  
8. 📈 Evaluation & Model Curves  
9. 🤖 SHAP Feature Importance  
10. 🧪 Interactive Prediction Demo  
11. 💾 Export & Download Model Artifacts  

> **💡 Tip:** Run all cells with `Runtime > Run all` or `Ctrl+F9`

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install any missing packages for Colab
!pip install shap --quiet
print("✅ All packages ready.")

In [ ]:
import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier,
    GradientBoostingRegressor
)
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    classification_report, roc_auc_score, f1_score,
    roc_curve, precision_recall_curve, confusion_matrix
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
np.random.seed(42)

print("✅ All libraries imported successfully!")
print(f"  numpy : {np.__version__}")
print(f"  pandas : {pd.__version__}")
import sklearn; print(f"  scikit-learn : {sklearn.__version__}")

## 🗃️ Step 2: Generate Synthetic Workforce Dataset

Since we are running on Google Colab, we generate a realistic synthetic dataset that mirrors the original enterprise data ($N = 178,750$ records) using statistically grounded distributions.

In [ ]:
def generate_workforce_dataset(n_samples: int = 50000, random_state: int = 42) -> pd.DataFrame:
    """
    Generate a realistic synthetic workforce dataset.
    Features are statistically correlated to produce realistic burnout scores.
    """
    rng = np.random.default_rng(random_state)

    job_roles = ['Software Engineer', 'Data Analyst', 'Project Manager', 'HR Manager',
                 'Sales Executive', 'Marketing Specialist', 'DevOps Engineer',
                 'Product Manager', 'Data Scientist', 'UX Designer']
    work_modes = ['Remote', 'Hybrid', 'In-Office']
    company_sizes = ['Startup', 'SME', 'Large Enterprise', 'MNC']

    stress_level            = rng.normal(5.5, 2.0, n_samples).clip(1, 10)
    work_hours_per_week     = rng.normal(42, 8, n_samples).clip(20, 80)
    overtime_hours          = rng.exponential(4, n_samples).clip(0, 25)
    meetings_per_day        = rng.integers(1, 10, n_samples).astype(float)
    deadlines_missed        = rng.integers(0, 6, n_samples).astype(float)
    sleep_hours             = rng.normal(6.5, 1.0, n_samples).clip(3, 9)
    physical_activity_days  = rng.integers(0, 7, n_samples).astype(float)
    job_satisfaction        = rng.normal(6.0, 2.0, n_samples).clip(1, 10)
    manager_support         = rng.normal(6.0, 2.0, n_samples).clip(1, 10)
    work_life_balance       = rng.normal(5.5, 2.0, n_samples).clip(1, 10)
    social_support_score    = rng.normal(6.5, 2.0, n_samples).clip(1, 10)
    screen_time_hours       = rng.normal(9, 2.5, n_samples).clip(2, 16)
    caffeine_intake         = rng.integers(0, 8, n_samples).astype(float)
    experience_years        = rng.integers(0, 30, n_samples).astype(float)

    job_role     = rng.choice(job_roles, n_samples)
    work_mode    = rng.choice(work_modes, n_samples, p=[0.35, 0.40, 0.25])
    company_size = rng.choice(company_sizes, n_samples, p=[0.15, 0.30, 0.30, 0.25])

    # Physics-based burnout score formula
    burnout_score = (
        0.30 * stress_level +
        0.18 * (work_hours_per_week / 8.0) +
        0.15 * (overtime_hours / 2.5) +
        0.10 * (meetings_per_day / 1.0) +
        0.08 * (deadlines_missed / 0.6) +
        0.06 * (screen_time_hours / 1.6) -
        0.12 * sleep_hours -
        0.10 * job_satisfaction -
        0.09 * manager_support -
        0.08 * work_life_balance -
        0.05 * (physical_activity_days / 0.7) +
        rng.normal(0, 0.4, n_samples)
    )

    # Normalize to [0, 10]
    burnout_score = (
        (burnout_score - burnout_score.min()) /
        (burnout_score.max() - burnout_score.min())
    ) * 10.0

    df = pd.DataFrame({
        'stress_level': stress_level.round(2),
        'work_hours_per_week': work_hours_per_week.round(1),
        'overtime_hours': overtime_hours.round(1),
        'meetings_per_day': meetings_per_day,
        'deadlines_missed': deadlines_missed,
        'sleep_hours': sleep_hours.round(1),
        'physical_activity_days': physical_activity_days,
        'job_satisfaction': job_satisfaction.round(2),
        'manager_support': manager_support.round(2),
        'work_life_balance': work_life_balance.round(2),
        'social_support_score': social_support_score.round(2),
        'screen_time_hours': screen_time_hours.round(1),
        'caffeine_intake': caffeine_intake,
        'experience_years': experience_years,
        'job_role': job_role,
        'work_mode': work_mode,
        'company_size': company_size,
        'burnout_score': burnout_score.round(3)
    })
    return df

# Generate Dataset
N_SAMPLES = 50000  # Increase to 178750 for full dataset (slower)
df = generate_workforce_dataset(n_samples=N_SAMPLES)

print(f"✅ Dataset generated: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nBurnout Score Stats:")
print(df['burnout_score'].describe().round(3))
display(df.head())

## 🔧 Step 3: Composite Feature Engineering

Creating four domain-specific composite features:

$$\text{Workload Index} = (0.4 \times H_{norm} + 0.3 \times O_{norm} + 0.2 \times M_{norm} + 0.1 \times D_{norm}) \times 100$$

$$\text{Wellness Score} = (0.3 \times S_{norm} + 0.2 \times A_{norm} + 0.3 \times B_{norm} + 0.2 \times J_{norm}) \times 100$$

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    """Compute composite domain features for improved predictive power."""
    df_eng = data.copy()

    # 1. Workload Index (higher = more overloaded)
    df_eng['workload_index'] = (
        (df_eng['work_hours_per_week'] / 40.0) * 0.4 +
        (df_eng['overtime_hours'] / 10.0) * 0.3 +
        (df_eng['meetings_per_day'] / 5.0) * 0.2 +
        (df_eng['deadlines_missed'] / 3.0) * 0.1
    ) * 100.0

    # 2. Wellness Score (higher = better wellbeing)
    sleep_norm = (df_eng['sleep_hours'] / 8.0).clip(0, 1.2)
    df_eng['wellness_score'] = (
        sleep_norm * 0.3 +
        (df_eng['physical_activity_days'] / 7.0) * 0.2 +
        (df_eng['work_life_balance'] / 10.0) * 0.3 +
        (df_eng['job_satisfaction'] / 10.0) * 0.2
    ) * 100.0

    # 3. Support Index (manager support scaled to 0-100)
    df_eng['support_index'] = df_eng['manager_support'] * 10.0

    # 4. Meeting Load Ratio (relative to 3 meetings/day baseline)
    df_eng['meeting_load'] = df_eng['meetings_per_day'] / 3.0

    return df_eng

df = engineer_features(df)
print("✅ Composite features added:")
display(df[['workload_index', 'wellness_score', 'support_index', 'meeting_load']].describe().round(2))

## 📊 Step 4: Exploratory Data Analysis (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Burnout Score Distribution
ax = axes[0, 0]
sns.histplot(df['burnout_score'], kde=True, color='#1a6fc4', ax=ax)
ax.axvline(4.5, color='#e84040', linestyle='--', linewidth=2, label='Risk Threshold (τ=4.5)')
ax.set_title('Burnout Score Distribution', fontweight='bold')
ax.legend()

# 2. Stress Level vs Burnout Score
ax = axes[0, 1]
sample = df.sample(3000, random_state=42)
scatter = ax.scatter(sample['stress_level'], sample['burnout_score'],
                     c=sample['burnout_score'], cmap='RdYlGn_r', alpha=0.5, s=10)
plt.colorbar(scatter, ax=ax, label='Burnout Score')
ax.set_title('Stress Level vs. Burnout Score', fontweight='bold')
ax.set_xlabel('Stress Level'); ax.set_ylabel('Burnout Score')

# 3. Burnout Score by Work Mode
ax = axes[0, 2]
sns.boxplot(data=df, x='work_mode', y='burnout_score', palette='Set2', ax=ax)
ax.set_title('Burnout Score by Work Mode', fontweight='bold')

# 4. Burnout Score by Job Role
ax = axes[1, 0]
role_order = df.groupby('job_role')['burnout_score'].mean().sort_values(ascending=False).index
sns.barplot(data=df, y='job_role', x='burnout_score', order=role_order, color='#0f2d56', ax=ax)
ax.set_title('Average Burnout Score by Job Role', fontweight='bold')

# 5. Workload Index vs Wellness Score (colored by burnout)
ax = axes[1, 1]
sc = ax.scatter(sample['workload_index'], sample['wellness_score'],
                c=sample['burnout_score'], cmap='RdYlGn_r', alpha=0.5, s=10)
plt.colorbar(sc, ax=ax, label='Burnout Score')
ax.set_title('Workload Index vs. Wellness Score', fontweight='bold')
ax.set_xlabel('Workload Index'); ax.set_ylabel('Wellness Score')

# 6. Elevated Risk % by Company Size
ax = axes[1, 2]
risk_by_company = df.groupby('company_size').apply(lambda x: (x['burnout_score'] >= 4.5).mean() * 100)
risk_by_company.sort_values(ascending=False).plot(kind='bar', color='#c0392b', ax=ax)
ax.set_title('Elevated Risk % by Company Size', fontweight='bold')
ax.set_ylabel('% Elevated Risk Employees')
ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Correlation Heatmap
numeric_cols_heatmap = [
    'stress_level', 'work_hours_per_week', 'overtime_hours', 'meetings_per_day',
    'sleep_hours', 'job_satisfaction', 'manager_support', 'work_life_balance',
    'workload_index', 'wellness_score', 'burnout_score'
]
plt.figure(figsize=(12, 8))
corr = df[numeric_cols_heatmap].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, annot_kws={'size': 9})
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## ✂️ Step 5: Preprocessing & Train/Test Split

In [ ]:
# Feature columns
NUMERIC_FEATURES = [
    'caffeine_intake', 'deadlines_missed', 'experience_years',
    'job_satisfaction', 'manager_support', 'meetings_per_day',
    'overtime_hours', 'physical_activity_days', 'screen_time_hours',
    'sleep_hours', 'social_support_score', 'stress_level',
    'work_hours_per_week', 'work_life_balance',
    'workload_index', 'wellness_score', 'support_index', 'meeting_load'
]
CATEGORICAL_FEATURES = ['company_size', 'job_role', 'work_mode']
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

THRESHOLD = 4.5  # Elevated risk threshold

X = df[ALL_FEATURES]
y_score = df['burnout_score']
y_risk  = (y_score >= THRESHOLD).astype(int)

X_train, X_test, y_train_score, y_test_score, y_train_risk, y_test_risk = train_test_split(
    X, y_score, y_risk, test_size=0.20, random_state=42, stratify=y_risk
)

# Preprocessing Pipelines
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
categorical_transformer = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES)
])

risk_pct = y_risk.mean() * 100
print(f"✅ Split complete")
print(f"  Training samples : {X_train.shape[0]:,}")
print(f"  Testing samples  : {X_test.shape[0]:,}")
print(f"  Features         : {X.shape[1]}")
print(f"  Elevated Risk %  : {risk_pct:.1f}%")

## 🎯 Step 6: Task A — Burnout Score Regression
Predicts a **continuous burnout score** $y \in [0, 10]$ using `HistGradientBoostingRegressor`.

In [ ]:
# Build & Train Regression Pipeline
regressor_pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_depth=8,
        min_samples_leaf=20,
        l2_regularization=0.1,
        random_state=42,
        verbose=0
    ))
])

print("⏳ Training HistGradientBoostingRegressor...")
regressor_pipeline.fit(X_train, y_train_score)

y_pred_score = np.clip(regressor_pipeline.predict(X_test), 0, 10)

mae  = mean_absolute_error(y_test_score, y_pred_score)
rmse = np.sqrt(mean_squared_error(y_test_score, y_pred_score))
r2   = r2_score(y_test_score, y_pred_score)

print(f"\n📊 Task A — Regression Metrics")
print(f"{'='*40}")
print(f"  Mean Absolute Error (MAE)   : {mae:.4f}")
print(f"  Root Mean Squared Error     : {rmse:.4f}")
print(f"  R² Score                    : {r2:.4f}")
print(f"{'='*40}")

# Residuals Plot
residuals = y_test_score.values - y_pred_score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_pred_score, y_test_score, alpha=0.3, s=8, color='#1a6fc4')
axes[0].plot([0, 10], [0, 10], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_title('Predicted vs Actual Burnout Score', fontweight='bold')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
axes[0].legend()
axes[0].text(0.05, 0.93, f'R² = {r2:.3f}  MAE = {mae:.3f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(facecolor='white', alpha=0.8))

sns.histplot(residuals, kde=True, color='#b95f00', ax=axes[1])
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Residuals Distribution', fontweight='bold')
axes[1].set_xlabel('Residual (Actual - Predicted)')

plt.tight_layout()
plt.show()

## 🚨 Step 7: Task B — Calibrated Elevated Risk Classification

Classifies employees as **elevated risk** (burnout score $\geq \tau = 4.5$)  
using `CalibratedClassifierCV` for reliable probability estimates.

In [ ]:
# Shared preprocessor clone for classifier
from sklearn.base import clone
preprocessor_clf = clone(preprocessor)

base_clf = Pipeline([
    ('prep', preprocessor_clf),
    ('model', HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.05,
        max_depth=8,
        min_samples_leaf=20,
        random_state=42
    ))
])

calibrated_clf = CalibratedClassifierCV(estimator=base_clf, method='sigmoid', cv=3)

print("⏳ Training CalibratedClassifierCV...")
calibrated_clf.fit(X_train, y_train_risk)

y_pred_prob  = calibrated_clf.predict_proba(X_test)[:, 1]
y_pred_class = (y_pred_prob >= 0.5).astype(int)

roc_auc = roc_auc_score(y_test_risk, y_pred_prob)
f1      = f1_score(y_test_risk, y_pred_class)

print(f"\n📊 Task B — Classification Metrics")
print(f"{'='*40}")
print(f"  ROC-AUC Score  : {roc_auc:.4f}")
print(f"  F1-Score       : {f1:.4f}")
print(f"{'='*40}")
print()
print(classification_report(
    y_test_risk, y_pred_class,
    target_names=['Standard Risk', 'Elevated Risk']
))

## 📈 Step 8: Evaluation Curves & Confusion Matrix

In [ ]:
fpr, tpr, _  = roc_curve(y_test_risk, y_pred_prob)
prec, rec, _ = precision_recall_curve(y_test_risk, y_pred_prob)
cm = confusion_matrix(y_test_risk, y_pred_class)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
axes[0].plot(fpr, tpr, color='#1a6fc4', lw=2, label=f'ROC Curve (AUC = {roc_auc:.3f})')
axes[0].plot([0, 1], [0, 1], 'gray', linestyle='--')
axes[0].fill_between(fpr, tpr, alpha=0.1, color='#1a6fc4')
axes[0].set_title('ROC Curve', fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# Precision-Recall Curve
axes[1].plot(rec, prec, color='#b95f00', lw=2, label='PR Curve')
axes[1].fill_between(rec, prec, alpha=0.1, color='#b95f00')
axes[1].set_title('Precision-Recall Curve', fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend()

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Standard', 'Elevated'], yticklabels=['Standard', 'Elevated'])
axes[2].set_title('Confusion Matrix', fontweight='bold')
axes[2].set_xlabel('Predicted Label')
axes[2].set_ylabel('True Label')

plt.tight_layout()
plt.show()

## 🤖 Step 9: SHAP Feature Importance
Using SHAP (SHapley Additive exPlanations) to interpret the regression model.

In [ ]:
print("⏳ Computing SHAP values (this may take 1-2 minutes)...")

# Get the preprocessed data for SHAP
prep_fitted = regressor_pipeline.named_steps['prep']
model_fitted = regressor_pipeline.named_steps['model']

# Use a sample for SHAP computation speed
X_shap = X_test.sample(min(2000, len(X_test)), random_state=42)
X_shap_transformed = prep_fitted.transform(X_shap)

# Get feature names after OHE
ohe_feature_names = prep_fitted.named_transformers_['cat'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERIC_FEATURES + ohe_feature_names

X_shap_df = pd.DataFrame(X_shap_transformed, columns=all_feature_names)

explainer = shap.TreeExplainer(model_fitted)
shap_values = explainer.shap_values(X_shap_df)

print("✅ SHAP values computed!")

# SHAP Summary Plot
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, X_shap_df, plot_type='bar',
                  max_display=15, show=False)
plt.title('SHAP Feature Importance (Global)', fontweight='bold')
plt.tight_layout()
plt.show()

# SHAP Dot Plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_shap_df, max_display=15, show=False)
plt.title('SHAP Values — Feature Impact on Burnout Score', fontweight='bold')
plt.tight_layout()
plt.show()

## 🧪 Step 10: Interactive Prediction Demo

Modify the **employee profile** below and run the cell to get real-time burnout predictions!

In [ ]:
# ============================================================
# 🎛️ EDIT THIS EMPLOYEE PROFILE
# ============================================================
employee_profile = {
    'stress_level'           : 7.5,   # 1–10 (higher = more stress)
    'work_hours_per_week'    : 52,    # hours per week
    'overtime_hours'         : 10,    # extra hours
    'meetings_per_day'       : 5,     # number of meetings
    'deadlines_missed'       : 2,     # count
    'sleep_hours'            : 5.5,   # hours of sleep per night
    'physical_activity_days' : 1,     # days per week with exercise
    'job_satisfaction'       : 4.0,   # 1–10
    'manager_support'        : 4.5,   # 1–10
    'work_life_balance'      : 3.0,   # 1–10
    'social_support_score'   : 5.0,   # 1–10
    'screen_time_hours'      : 11,    # hours per day
    'caffeine_intake'        : 4,     # cups per day
    'experience_years'       : 5,
    'job_role'               : 'Software Engineer',
    'work_mode'              : 'Remote',
    'company_size'           : 'MNC',
}
# ============================================================

def predict_employee_burnout(profile: dict):
    """Run prediction for a single employee profile."""
    # Fill missing features with dataset medians
    defaults = X.median(numeric_only=True).to_dict()
    defaults['company_size'] = X['company_size'].mode()[0]
    defaults['job_role']     = X['job_role'].mode()[0]
    defaults['work_mode']    = X['work_mode'].mode()[0]
    defaults.update(profile)

    row_df = pd.DataFrame([defaults])
    row_df = engineer_features(row_df)
    row_df = row_df[ALL_FEATURES]

    score = float(regressor_pipeline.predict(row_df)[0])
    score = np.clip(score, 0, 10)
    prob  = float(calibrated_clf.predict_proba(row_df)[0, 1]) * 100.0

    # Risk bands
    if score >= 7.5:
        band, emoji, color = 'CRITICAL', '🔴', '#e74c3c'
    elif score >= 5.5:
        band, emoji, color = 'HIGH', '🟠', '#e67e22'
    elif score >= 4.5:
        band, emoji, color = 'ELEVATED', '🟡', '#f1c40f'
    elif score >= 2.5:
        band, emoji, color = 'MODERATE', '🟢', '#2ecc71'
    else:
        band, emoji, color = 'LOW', '✅', '#27ae60'

    print('\n' + '='*55)
    print(f'  🎯 BURNOUT RISK ASSESSMENT REPORT')
    print('='*55)
    print(f'  Burnout Score (0–10)        : {score:.2f} / 10')
    print(f'  Elevated Risk Probability   : {prob:.1f}%')
    print(f'  Risk Band                   : {emoji} {band}')
    print('='*55)
    print()
    print('  📌 Recommendations:')
    if score >= 6:
        print('  • 🚨 Immediate HR intervention recommended')
        print('  • 📅 Schedule 1:1 with manager this week')
        print('  • 🛌 Enforce minimum 7h sleep goal')
        print('  • 🏃 Encourage physical activity schedule')
    elif score >= 4.5:
        print('  • ⚠️ Monitor employee closely over next 30 days')
        print('  • 💬 Check-in meeting with manager')
        print('  • 📉 Reduce overtime if possible')
    else:
        print('  • ✅ Employee wellbeing appears satisfactory')
        print('  • 🔄 Continue regular check-ins')
    print('='*55)

predict_employee_burnout(employee_profile)

## 💾 Step 11: Export & Download Model Artifacts

In [ ]:
import zipfile

os.makedirs('artifacts', exist_ok=True)

# Save models
joblib.dump(regressor_pipeline,  'artifacts/burnout_score_model.joblib')
joblib.dump(calibrated_clf,      'artifacts/elevated_risk_model.joblib')

# Save metrics
metrics = {
    'regressor' : {'mae': round(mae,4), 'rmse': round(rmse,4), 'r2': round(r2,4)},
    'risk_model': {'threshold': THRESHOLD, 'roc_auc': round(roc_auc,4), 'f1': round(f1,4)},
    'dataset'   : {'n_samples': N_SAMPLES, 'n_features': len(ALL_FEATURES)}
}
with open('artifacts/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✅ Artifacts saved:")
for fname in os.listdir('artifacts'):
    fsize = os.path.getsize(f'artifacts/{fname}')
    print(f"  {fname:40s}  {fsize:>10,} bytes")

# Zip everything for download
with zipfile.ZipFile('burnout_model_artifacts.zip', 'w') as z:
    for fname in os.listdir('artifacts'):
        z.write(f'artifacts/{fname}', fname)

print("\n✅ burnout_model_artifacts.zip created.")

In [ ]:
# Download the zip file directly from Colab
try:
    from google.colab import files
    files.download('burnout_model_artifacts.zip')
    print("✅ Download started!")
except ImportError:
    print("ℹ️ Not running in Colab — artifacts are in the 'artifacts/' folder.")

---

## 🎉 Pipeline Complete!

| Step | Module | Status |
|---|---|---|
| Data Generation | Synthetic N=50,000 | ✅ |
| Feature Engineering | 4 composite features | ✅ |
| EDA & Visualization | 6 charts + correlation heatmap | ✅ |
| Task A (Regression) | HistGradientBoostingRegressor | ✅ |
| Task B (Classification) | CalibratedClassifierCV | ✅ |
| Model Evaluation | ROC, PR, Confusion Matrix | ✅ |
| SHAP Explainability | TreeExplainer | ✅ |
| Prediction Demo | Interactive inference | ✅ |
| Model Export | .joblib + metrics.json + .zip | ✅ |

**GitHub**: https://github.com/ShivanshuRai07/Burnout
